# Fine-tune FunctionGemma-270M for PRISM

Trains the local tool-routing model on Google's [Mobile Actions](https://huggingface.co/datasets/google/mobile-actions). With Colab Pro's A100 this finishes in ~15 minutes.

**Outputs (private repos under your account):**
- `Darth-Hidious/functiongemma-prism-lora` — LoRA adapter checkpoints
- `Darth-Hidious/functiongemma-prism-merged` — merged HF model
- `Darth-Hidious/functiongemma-prism-gguf` — Q4_K_M GGUF (what PRISM downloads)

## Critical correctness notes
Earlier failed runs trained on raw text, which made the model overfit to dataset prompts and break inference. This notebook uses:
- chat-format dataset (`messages` + `tools` per row, no pre-rendered text)
- `assistant_only_loss=True` so loss only flows through the assistant's tool_call output, not the prompt
- conservative LR (1e-4) since masked loss has higher gradient variance

## To run
1. **Runtime → Change runtime type → A100** (Colab Pro)
2. **Runtime → Run all**
3. When cell 2 prompts, paste your HF token (write scope, from https://huggingface.co/settings/tokens)

## 1. Install

In [ ]:
%%capture
!pip install --quiet --upgrade unsloth datasets transformers "trl>=0.13.0" peft accelerate huggingface_hub
!pip install --quiet gguf>=0.10.0 mistral_common sentencepiece

## 2. Authenticate

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Load FunctionGemma + LoRA

In [ ]:
from unsloth import FastLanguageModel
BASE_MODEL = 'unsloth/functiongemma-270m-it'
MAX_LEN = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_LEN,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing='unsloth', random_state=42,
)
print('model + LoRA ready')

## 4. Load Mobile Actions in CHAT FORMAT (no pre-rendering)

In [ ]:
import json
from datasets import load_dataset

raw = load_dataset('google/mobile-actions', split='train')

def normalise_messages(messages):
    out = []
    for m in messages:
        role = m['role']
        if role == 'assistant' and m.get('tool_calls'):
            calls = []
            for c in m['tool_calls']:
                fn = c['function']
                args = fn.get('arguments', {}) or {}
                if isinstance(args, dict):
                    args = {k: v for k, v in args.items() if v is not None}
                calls.append({'type': 'function', 'function': {'name': fn['name'], 'arguments': json.dumps(args, default=str)}})
            out.append({'role': 'assistant', 'content': '', 'tool_calls': calls})
        else:
            out.append({'role': role, 'content': m.get('content') or ''})
    return out

def reformat(row):
    return {'messages': normalise_messages(row['messages']), 'tools': row['tools']}

train_rows = raw.filter(lambda r: r['metadata'] == 'train').map(reformat, remove_columns=raw.column_names)
eval_rows  = raw.filter(lambda r: r['metadata'] == 'eval').map(reformat,  remove_columns=raw.column_names)
print(f'train: {len(train_rows)}, eval: {len(eval_rows)}')

## 5. Train with assistant_only_loss

In [ ]:
from trl import SFTConfig, SFTTrainer
HUB_REPO_LORA = 'Darth-Hidious/functiongemma-prism-lora'
args = SFTConfig(
    output_dir='/tmp/fgp',
    max_length=MAX_LEN,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=25,
    eval_strategy='steps', eval_steps=200,
    save_strategy='steps', save_steps=200, save_total_limit=3,
    bf16=True,
    optim='adamw_torch_fused',
    report_to='none',
    push_to_hub=True, hub_model_id=HUB_REPO_LORA, hub_strategy='every_save', hub_private_repo=True,
    seed=42,
    assistant_only_loss=True,  # CRITICAL — fixes the prompt-overfit bug
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_rows, eval_dataset=eval_rows, args=args)
trainer.train()
trainer.push_to_hub()
print(f'LoRA → {HUB_REPO_LORA}')

## 6. Merge + push HF model

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path
HUB_REPO_MERGED = 'Darth-Hidious/functiongemma-prism-merged'
merged = model.merge_and_unload()
merged.save_pretrained('/tmp/fgp-merged')
tokenizer.save_pretrained('/tmp/fgp-merged')
api = HfApi()
api.create_repo(repo_id=HUB_REPO_MERGED, private=True, exist_ok=True)
api.upload_folder(folder_path='/tmp/fgp-merged', repo_id=HUB_REPO_MERGED, commit_message='merged FunctionGemma + Mobile Actions LoRA (mask-prompt)')
print(f'merged → {HUB_REPO_MERGED}')

## 7. Convert to GGUF Q4_K_M + push (PRISM downloads from here)

In [ ]:
import urllib.request, subprocess, shutil
from pathlib import Path
from huggingface_hub import HfApi

HUB_REPO_GGUF = 'Darth-Hidious/functiongemma-prism-gguf'
work = Path('/tmp/fgp-work'); work.mkdir(exist_ok=True)
convert = work / 'convert_hf_to_gguf.py'
urllib.request.urlretrieve('https://raw.githubusercontent.com/ggml-org/llama.cpp/b6700/convert_hf_to_gguf.py', convert)
f16 = work / 'functiongemma-prism-f16.gguf'
subprocess.run(['python', str(convert), '/tmp/fgp-merged', '--outfile', str(f16), '--outtype', 'f16'], check=True)

# Build llama.cpp's quantize binary if not present (Colab nodes don't have it)
import os
if not shutil.which('llama-quantize'):
    print('compiling llama.cpp quantize...')
    !git clone --depth=1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp >/dev/null 2>&1
    !cd /tmp/llama.cpp && cmake -B build -DGGML_NATIVE=OFF >/dev/null 2>&1 && cmake --build build --config Release --target llama-quantize -j 4 >/dev/null 2>&1
    quantize_bin = '/tmp/llama.cpp/build/bin/llama-quantize'
else:
    quantize_bin = shutil.which('llama-quantize')

q4 = work / 'functiongemma-prism-Q4_K_M.gguf'
subprocess.run([quantize_bin, str(f16), str(q4), 'Q4_K_M'], check=True)

api = HfApi()
api.create_repo(repo_id=HUB_REPO_GGUF, private=True, exist_ok=True)
api.upload_file(path_or_fileobj=str(q4), path_in_repo=q4.name, repo_id=HUB_REPO_GGUF, commit_message='initial fine-tune')
print(f'GGUF → {HUB_REPO_GGUF}')
print()
print('DONE. On your local PRISM machine, set:')
print('  export PRISM_FUNCTION_REPO=Darth-Hidious/functiongemma-prism-gguf')
print(f'  export PRISM_FUNCTION_FILE={q4.name}')
print('  rm ~/.prism/models/functiongemma-270m.gguf')
print('  prism tui   # auto-downloads the fine-tuned weights from your repo')